## Tugas Crawling Website Berita (Sumber dari CNN Indonesia)

### import library
Tahap ini menyiapkan alat bantu berupa library Python. requests digunakan untuk mengambil halaman web, BeautifulSoup untuk membaca struktur HTML, pandas untuk menyimpan hasil ke dalam tabel, dan random untuk memilih artikel secara acak.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random

### Fungsi Pengambil Judul Artikel
Fungsi ini bertugas memastikan judul artikel bisa diambil walaupun CNN Indonesia memakai beberapa variasi format tampilan. Jika tidak ditemukan, maka ditandai dengan teks "Tidak ada judul".

In [2]:
def get_judul(soup):
    judul_tag = soup.find("h1", class_="mb-2 text-[28px] leading-9 text-cnn_black")
    if not judul_tag:
        judul_tag = soup.find("h1", class_="mb-2 text-[32px] text-cnn_black font-merriweather")
    if not judul_tag:
        judul_tag = soup.find("h1", class_="text-2xl text-white inline")
    return judul_tag.get_text(strip=True) if judul_tag else "Tidak ada judul"

### Fungsi Scraping Artikel CNN
Di sini program mengambil isi artikel sesuai dengan struktur halaman CNN. Ada tiga mode:

* Artikel normal (dari div detail-text).

* Artikel singkat (langsung setelah teks pembuka "Jakarta, CNN Indonesia").

* Artikel live report (khusus olahraga, berisi beberapa update dalam bentuk daftar).
Hasilnya berupa judul, kategori, isi, dan link artikel.

In [3]:
def scrape_cnn(url, kategori):
    try:
        response = requests.get(url)
        response.raise_for_status()
    except:
        print("Gagal akses:", url)
        return None

    if "/longform/" in response.url:
        print("Skip longform:", response.url)
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # --- Judul ---
    judul = get_judul(soup)

    isi = "Tidak ada isi"

    # --- Mode 1: Artikel normal (detail-text) ---
    detail_div = soup.find("div", class_=lambda x: x and "detail-text" in x)
    if detail_div:
        end_div = detail_div.find("div", class_="end-of-article")
        if end_div:
            end_div.decompose()
        isi = detail_div.get_text(" ", strip=True)

    # --- Mode 2: Artikel singkat (CNN Indonesia intro) ---
    if isi == "Tidak ada isi":
        strong_tag = soup.find("strong")
        if strong_tag and "CNN Indonesia" in strong_tag.get_text():
            parent_div = strong_tag.find_parent("div")
            if parent_div:
                isi = parent_div.get_text(" ", strip=True)

    # --- Mode 3: Live report (olahraga) ---
    if isi == "Tidak ada isi":
        live_report = soup.find("ul", class_="flex flex-col pl-5")
        if live_report:
            updates = []
            for li in live_report.find_all("li"):
                title = li.find("h2")
                content = li.find("p")
                if title or content:
                    updates.append(f"{title.get_text(strip=True) if title else ''} {content.get_text(strip=True) if content else ''}")
            isi = "\n".join(updates) if updates else "Tidak ada isi"

    return {
        "Judul": judul,
        "Kategori": kategori.replace("-", " ").title(),
        "Isi": isi,
        "URL": url
    }


### Fungsi Pengambil Artikel Random dari Satu Kategori
Fungsi ini mengambil daftar link berita dari halaman kategori CNN, lalu memilih beberapa secara acak. Link yang valid kemudian diproses dengan fungsi scraping agar isinya bisa diambil.

In [4]:
def get_random_cnn_berita(kategori="nasional", jumlah=50):
    url = f"https://www.cnnindonesia.com/{kategori}"
    try:
        response = requests.get(url)
        response.raise_for_status()
    except:
        print("Gagal akses kategori:", kategori)
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # --- Cari semua link artikel ---
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if (f"/{kategori}/" in href
            and "indeks/" not in href
            and "/longform/" not in href
            and any(char.isdigit() for char in href)):
            if href.startswith("http"):
                links.append(href)
            else:
                links.append("https://www.cnnindonesia.com" + href)

    if not links:
        return []

    # --- Ambil random N artikel ---
    random_links = random.sample(links, min(jumlah, len(links)))

    hasil = []
    for link in random_links:
        berita_satu = scrape_cnn(link, kategori)
        if berita_satu:  # hanya append jika bukan None
            hasil.append(berita_satu)

    return hasil

### Main Program
Bagian ini menjalankan keseluruhan proses: menentukan kategori, mengambil artikel acak dari masing-masing kategori, menggabungkan semua hasil, lalu menyimpannya ke dalam tabel. Hasil akhir ditampilkan sebagian dan disimpan ke file Excel serta CSV.

In [5]:
# Daftar kategori
kategori_list = [
    "nasional",
    "internasional",
    "ekonomi",
    "olahraga",
    "teknologi",
    "otomotif",
    "hiburan",
    "gaya-hidup"
]

# Proses scraping semua kategori
semua_berita = []
for kategori in kategori_list:
    berita = get_random_cnn_berita(kategori, jumlah=50)
    semua_berita.extend(berita)

# Buat DataFrame
df = pd.DataFrame(semua_berita)

# Simpan hasil ke file
df.to_excel("cnn_berita.xlsx", index=False)
df.to_csv("cnn_berita.csv", index=False)


Skip longform: https://www.cnnindonesia.com/longform/gaya-hidup/20241108/longform-balada-kerupuk-negeri-singkong/index.html


Tampilkan Hasil Crawling Berita CNN Indonesia

In [6]:
import pandas as pd

# Baca file
df = pd.read_csv("cnn_berita.csv")

# Tampilkan 10 data pertama
# df.head(10)

# Atau kalau ingin 10 data acak
df.sample(10)


,Judul,Kategori,Isi,URL
136,Pakar Prediksi iPhone Air Bisa Dongkrak Penjua...,Teknologi,"Jakarta, CNN Indonesia -- Apple membuat gebrak...",https://www.cnnindonesia.com/teknologi/2025091...
23,Update Korban Banjir di Bali: 2 Dilaporkan Tew...,Nasional,"Denpasar, CNN Indonesia -- Kepala Kantor Penca...",https://www.cnnindonesia.com/nasional/20250910...
137,"Resmi Meluncur, Intip Perkiraan Harga iPhone 1...",Teknologi,"Jakarta, CNN Indonesia -- Apple resmi merilis ...",https://www.cnnindonesia.com/teknologi/2025091...
149,Sri Mulyani Pernah Bilang Ingin Mobil Ini saat...,Otomotif,"Jakarta, CNN Indonesia -- Sri Mulyani telah re...",https://www.cnnindonesia.com/otomotif/20250910...
1,Prabowo Berduka Banjir Terjang Wilayah Bali da...,Nasional,"Jakarta, CNN Indonesia -- Presiden RI Prabowo ...",https://www.cnnindonesia.com/nasional/20250910...
103,Verona vs Cremonese: Audero dkk Berpeluang Tet...,Olahraga,"Jakarta, CNN Indonesia -- Cremonese yang diper...",https://www.cnnindonesia.com/olahraga/20250910...
221,FOTO: Kolektor Memorabilia Super Mario Terbany...,Gaya Hidup,"Jakarta, CNN Indonesia --\n ...",https://www.cnnindonesia.com/gaya-hidup/202509...
161,"Isi Garasi Menteri Haji dan Umrah Gus Irfan, C...",Otomotif,"Jakarta, CNN Indonesia -- Mochammad Irfan Yusu...",https://www.cnnindonesia.com/otomotif/20250909...
143,"Spesifikasi Lengkap iPhone 17, Apa Saja yang B...",Teknologi,"Daftar Isi Kamera depan Center Stage Layar 6,3...",https://www.cnnindonesia.com/teknologi/2025091...
20,"VIDEO: Banjir Terjang Bali, Dua Orang Meningga...",Nasional,"Jakarta, CNN Indonesia -- Banjir besar melanda...",https://www.cnnindonesia.com/nasional/20250910...
